# TP Databricks — NYC Taxi Data Engineering

**Formation :** SUPINFO — 5LAKE Data Engineering  
**Projet :** NYC Taxi — Data Pipeline & Analytics  
**Responsabilité :** Partie 1 (Ingestion) et Partie 2 (Delta Lake)

## Contexte

Dans le cadre de ce projet, nous intervenons en tant que Data Engineers pour une société de VTC souhaitant analyser son activité et développer un modèle prédictif.

Notre mission consiste à préparer les données de courses NYC Taxi pour les étapes suivantes du projet : pipelines déclaratifs, analyses BI et Machine Learning.

## Objectifs

- Créer le schéma du projet et le Volume `raw`.
- Extraire les données NYC Taxi et les stocker au format Parquet.
- Explorer la structure et la qualité des données.
- Construire les tables Delta `bronze_trips` et `silver_trips`.
- Nettoyer les données et créer les indicateurs demandés.
- Appliquer des corrections avec `MERGE INTO`.
- Explorer l'historique Delta Lake et le Time Travel.
- Répondre à la question 1 du TP.

## 1.1. Création du schéma

Nous créons un schéma dédié au projet NYC Taxi.

Ce schéma regroupera les volumes et les tables Delta utilisés par les différentes parties du TP.

Le catalogue utilisé est `workspace` et le schéma du projet est `groupe_5lake_lab`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.5lake_lab;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
5lake_lab
default
information_schema


## 2 — Création du Volume raw

Nous créons un Volume nommé `raw` dans le schéma `5lake_lab`.

Ce Volume servira à stocker les fichiers Parquet issus de la table source `samples.nyctaxi.trips`, avant leur ingestion dans la couche Bronze.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.`5lake_lab`.raw;

In [0]:
%sql
SHOW VOLUMES IN workspace.`5lake_lab`;

database,volume_name
5lake_lab,raw


## 3 — Ingestion des données NYC Taxi

Nous utilisons la table `samples.nyctaxi.trips` comme source de données.

Le TP demande un échantillon d'environ 300 000 lignes. Cependant, la table accessible dans notre environnement contient 21 932 lignes. Nous utilisons donc toutes les données disponibles.

Les données sont exportées au format Parquet dans le Volume `raw`, sans transformation, afin de constituer notre source brute pour la couche Bronze.

In [0]:
source_table = "samples.nyctaxi.trips"
raw_path = "/Volumes/workspace/5lake_lab/raw/nyc_trips"

#lecture de la source
df_raw = spark.table(source_table)

#export en parquet
df_raw.write.format("parquet").mode("overwrite").save(raw_path)

print("Nombre de lignes exportées :", df_raw.count())
print("Chemin de stockage :", raw_path)

Nombre de lignes exportées : 21932
Chemin de stockage : /Volumes/workspace/5lake_lab/raw/nyc_trips


In [0]:
df_check = spark.read.parquet(raw_path)

print("Nombre de lignes relues :", df_check.count())
display(df_check.limit(5))

Nombre de lignes relues : 21932


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022


## 4 — Exploration et qualité des données

Cette étape consiste à examiner les données brutes avant leur ingestion dans la couche Bronze.

Nous analysons les valeurs manquantes et la distribution des variables `trip_distance` et `fare_amount`, afin d'identifier d'éventuelles anomalies dans les données de courses.

In [0]:
from pyspark.sql import functions as F

#on cherche a compter tout les NULL de chaque colonne
df_nulls = df_check.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_check.columns
])

display(df_nulls)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
0,0,0,0,0,0


In [0]:
#stats sur les deux colonnes visées
display(
    df_check.select(
        "trip_distance",
        "fare_amount"
    ).describe()
)

summary,trip_distance,fare_amount
count,21932,21932
mean,2.852829199343427,12.34872697428415
stddev,3.4399536210623523,10.327887859233329
min,0.0,-8.0
max,30.6,275.0


## 5 — Distribution des distances et des tarifs

Nous visualisons la répartition de `trip_distance` et `fare_amount` à l'aide d'histogrammes afin d'identifier la concentration des valeurs et les éventuelles valeurs extrêmes.

Les statistiques précédentes révèlent notamment des distances nulles et des tarifs négatifs, qui seront traités lors du nettoyage de la couche Silver.

In [0]:
display(
    df_check.select("trip_distance", "fare_amount")
)

trip_distance,fare_amount
1.4,8.0
1.31,7.5
1.8,9.5
2.3,11.5
2.6,18.5
1.4,6.5
10.4,31.0
10.15,28.5
3.27,15.0
4.42,15.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## 2. Delta Lake — Bronze & Silver
### 2.1 — Création de la couche Bronze

La couche Bronze contient les données brutes provenant du Volume `raw`.

Nous lisons les fichiers Parquet précédemment exportés, puis nous les enregistrons dans une table Delta Lake nommée `bronze_trips`, sans effectuer de transformation.

Cette table servira de source pour la création de la couche Silver.

In [0]:
raw_path = "/Volumes/workspace/5lake_lab/raw/nyc_trips"

df_bronze = spark.read.parquet(raw_path)

print("Nombre de lignes :", df_bronze.count())

Nombre de lignes : 21932


In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.5lake_lab.bronze_trips")

In [0]:
# on verifie la table bronze
df_bronze_check = spark.table(
    "workspace.`5lake_lab`.bronze_trips"
)

print("Nombre de lignes Bronze :", df_bronze_check.count())

df_bronze_check.printSchema()

Nombre de lignes Bronze : 21932
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



### 2.2 — Création de la couche Silver

La couche Silver est construite à partir de `bronze_trips` avec les transformations suivantes :

- Suppression des doublons.
- Exclusion des courses avec `trip_distance <= 0` ou `fare_amount <= 0`.
- Exclusion des courses dont la date de fin précède la date de début.
- Calcul de la durée des courses en minutes (`trip_duration_min`).
- Génération d'une colonne `tip_pct` simulée, comprise entre 0 % et 30 %.


In [0]:
from pyspark.sql import functions as F

# on lit la table bronze
df_bronze = spark.table("workspace.`5lake_lab`.bronze_trips")

# on applique nos règles de nettoyage
df_silver = (
    df_bronze
    .dropDuplicates()
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(
        F.col("tpep_dropoff_datetime") >=
        F.col("tpep_pickup_datetime")
    )
    .withColumn(
        "trip_duration_min",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60.0
    )
    .withColumn(
    "tip_pct",
    F.round(F.rand(seed=42) * 30, 2)
)
)

# Compare row counts
print("Bronze rows:", df_bronze.count())
print("Silver rows:", df_silver.count())

display(df_silver.limit(10))

Bronze rows: 21932
Silver rows: 21847


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,trip_duration_min,tip_pct
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110,9.366666666666667,2.57
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023,8.233333333333333,9.31
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018,11.566666666666666,1.88
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111,14.183333333333334,9.19
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022,30.616666666666667,0.0
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069,5.15,9.09
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003,23.0,2.63
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201,16.633333333333333,6.81
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023,19.266666666666666,25.88
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222,13.366666666666667,15.7


### 2.3 — Enregistrement de la table Silver

Après application des règles de nettoyage, les données sont enregistrées dans une table Delta Lake nommée `silver_trips`.

La table contient 21 847 lignes valides, contre 21 932 dans la couche Bronze, soit 85 lignes exclues.

La colonne `trip_duration_min` a été ajoutée. La colonne `tip_pct` ne peut pas être calculée, car les données sources ne contiennent pas de pourboire.

In [0]:
df_silver.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("workspace.`5lake_lab`.silver_trips")

In [0]:
#on verifie après l'avoir crée
df_silver_check = spark.table(
    "workspace.`5lake_lab`.silver_trips"
)

print("Silver rows:", df_silver_check.count())

df_silver_check.printSchema()

Silver rows: 21847
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- tip_pct: double (nullable = true)



In [0]:
# on crée une clé pour identifier les lignes
merge_keys = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_zip",
    "dropoff_zip"
]

nb_rows = df_silver_check.count()

nb_unique_keys = (
    df_silver_check.select(*merge_keys).distinct().count()
)

print("Nombre de lignes :", nb_rows)
print("Nombre de clés distinctes :", nb_unique_keys)
print("Différence :", nb_rows - nb_unique_keys)

Nombre de lignes : 21847
Nombre de clés distinctes : 21847
Différence : 0


### 2.4 — Correction des données avec MERGE INTO

Nous simulons la réception de 30 corrections tarifaires.

Un DataFrame contenant 30 courses existantes est créé, avec une augmentation fictive de 10 % du tarif `fare_amount`.

Ces corrections seront appliquées à la table Delta `silver_trips` avec `MERGE INTO`, en utilisant une clé composée dont l'unicité a été vérifiée.

L'objectif est de mettre à jour les enregistrements concernés sans reconstruire l'ensemble de la table.

In [0]:
from pyspark.sql import functions as F

df_corrections = (
    df_silver_check
    .orderBy(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pickup_zip",
        "dropoff_zip"
    )
    .limit(30)
    .withColumn(
        "fare_amount",
        F.round(F.col("fare_amount") * 1.10, 2)
    )
)

print("Nombre de corrections :", df_corrections.count())

display(
    df_corrections.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "fare_amount"
    )
)

Nombre de corrections : 30


tpep_pickup_datetime,tpep_dropoff_datetime,fare_amount
2016-01-01T00:04:30.000Z,2016-01-01T00:07:42.000Z,5.5
2016-01-01T00:11:29.000Z,2016-01-01T00:30:42.000Z,26.95
2016-01-01T00:12:48.000Z,2016-01-01T00:16:23.000Z,5.5
2016-01-01T00:13:37.000Z,2016-01-01T00:25:38.000Z,11.0
2016-01-01T00:15:46.000Z,2016-01-01T00:23:57.000Z,9.35
2016-01-01T00:15:57.000Z,2016-01-01T00:27:00.000Z,12.1
2016-01-01T00:20:57.000Z,2016-01-01T00:25:32.000Z,6.05
2016-01-01T00:22:30.000Z,2016-01-01T00:25:19.000Z,4.4
2016-01-01T00:23:34.000Z,2016-01-01T00:37:18.000Z,14.3
2016-01-01T00:24:29.000Z,2016-01-01T00:37:09.000Z,11.0


### 2.5 — Application des corrections avec MERGE INTO

Les 30 corrections tarifaires sont appliquées à la table Delta `silver_trips`.

La correspondance entre les courses est effectuée à l'aide d'une clé composée dont l'unicité a été vérifiée.

L'opération `MERGE INTO` met à jour uniquement la colonne `fare_amount` pour les courses correspondantes, sans ajouter de nouvelles lignes.

Cette opération crée une nouvelle version de la table Delta, consultable avec le Time Travel.

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "workspace.`5lake_lab`.silver_trips"
)

(
    silver_table.alias("target")
    .merge(
        df_corrections.alias("source"),
        """
        target.tpep_pickup_datetime = source.tpep_pickup_datetime
        AND target.tpep_dropoff_datetime = source.tpep_dropoff_datetime
        AND target.pickup_zip <=> source.pickup_zip
        AND target.dropoff_zip <=> source.dropoff_zip
        """
    )
    .whenMatchedUpdate(
        set={
            "fare_amount": "source.fare_amount"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# on veirife le nombre de ligne ssuite au merge
df_silver_after = spark.table(
    "workspace.`5lake_lab`.silver_trips"
)

print("Silver après MERGE:", df_silver_after.count())

display(
    spark.sql("""
        DESCRIBE HISTORY workspace.`5lake_lab`.silver_trips
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

Silver après MERGE: 21847


version,timestamp,operation,operationMetrics
3,2026-10-22T14:37:40.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4389, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 30, executionTimeMs -> 8045, materializeSourceTimeMs -> 532, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2748, numTargetRowsUpdated -> 30, numOutputRows -> 30, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 30, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4657)"
2,2026-10-22T14:32:57.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 355480, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 397671)"
1,2026-10-22T14:29:45.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 355480, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 355480)"
0,2026-10-22T14:29:38.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 355480)"


### 2.6 — Historique Delta et Time Travel

L'opération `MERGE INTO` a créé une nouvelle version de la table `silver_trips`.

Nous utilisons le Time Travel de Delta Lake pour consulter la version 2, correspondant à l'état de la table avant le MERGE, puis nous la comparons à la version 3.

Cette comparaison permet de vérifier les corrections tarifaires sans restaurer ni modifier la table actuelle.

In [0]:
#avant le merge
df_before_merge = (
    spark.read
    .format("delta")
    .option("versionAsOf", 2)
    .table("workspace.`5lake_lab`.silver_trips")
)

#après merge
df_after_merge = (
    spark.read
    .format("delta")
    .option("versionAsOf", 3)
    .table("workspace.`5lake_lab`.silver_trips")
)

print("Avant MERGE :", df_before_merge.count())
print("Après MERGE :", df_after_merge.count())

Avant MERGE : 21847
Après MERGE : 21847


In [0]:
# on compare ce q'on a modifié
keys = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_zip",
    "dropoff_zip"
]

df_comparison = (
    df_before_merge.alias("before")
    .join(
        df_after_merge.alias("after"),
        on=keys,
        how="inner"
    )
    .filter(
        F.col("before.fare_amount")
        != F.col("after.fare_amount")
    )
    .select(
        *keys,
        F.col("before.fare_amount").alias("fare_before"),
        F.col("after.fare_amount").alias("fare_after")
    )
)

print("Nombre de tarifs modifiés :", df_comparison.count())
display(df_comparison.limit(10))

Nombre de tarifs modifiés : 30


tpep_pickup_datetime,tpep_dropoff_datetime,pickup_zip,dropoff_zip,fare_before,fare_after
2016-01-01T00:57:49.000Z,2016-01-01T01:00:47.000Z,10023,10162,4.5,4.95
2016-01-01T00:36:40.000Z,2016-01-01T00:57:00.000Z,10022,10012,15.5,17.05
2016-01-01T00:15:46.000Z,2016-01-01T00:23:57.000Z,10007,10014,8.5,9.35
2016-01-01T00:12:48.000Z,2016-01-01T00:16:23.000Z,10035,10029,5.0,5.5
2016-01-01T00:11:29.000Z,2016-01-01T00:30:42.000Z,10002,10025,24.5,26.95
2016-01-01T00:41:52.000Z,2016-01-01T01:04:39.000Z,10028,11375,27.0,29.7
2016-01-01T00:35:19.000Z,2016-01-01T00:59:47.000Z,10028,10199,21.0,23.1
2016-01-01T00:51:30.000Z,2016-01-01T01:10:32.000Z,10026,10452,18.0,19.8
2016-01-01T00:23:34.000Z,2016-01-01T00:37:18.000Z,10103,10162,13.0,14.3
2016-01-01T00:20:57.000Z,2016-01-01T00:25:32.000Z,10003,10016,5.5,6.05


### 2.7 — Question 1 : Historique Delta Lake

**Que montre l’historique ?**

L'historique de `silver_trips` montre quatre versions. Les premières correspondent à la création et aux reconstructions de la table, tandis que la version 3 correspond à l'opération `MERGE`.

**Pourquoi Delta est-il préférable à du Parquet simple pour ce cas d’usage ?**

Delta Lake est préférable à de simples fichiers Parquet dans ce contexte, car il apporte notamment :
- Les transactions ACID
- Le MERGE INTO, pour modifier des enregistrements sans reconstruire tout le dataset.
- L'historique des versions, pour voir les modifs
- Le Time Travel, pour consulter rétroactivement les données

Ces fonctionnalités facilitent la traçabilité des corrections et la fiabilité des pipelines de données.